In [1]:
import os
import torch
import json
import numpy as np
import matplotlib.pyplot as plt
from itertools import islice
from cryo_sbi import CryoEmSimulator
from cryo_sbi.inference.models import build_models
import cryo_sbi.inference.train_nle_model as train_nle_model
import cryo_sbi.utils.estimator_utils as est_utils
from cryo_sbi.inference.priors import get_image_priors, PriorLoader
import cryo_sbi.utils.image_utils as img_utils
from cryo_sbi.utils.visualize_models import plot_model

### Read the models 

Here we work with the HSP90 chaperone. We load 20 pre-calculated models, that span a continuum between a close state (model 1) and an open state (model 20). Each model is composed by 1207 beads. The models are safely stored in the root directoy of these tutorials in `hsp90_models.pt`.

In [2]:
# load models from file
models = torch.load("../hsp90_models.pt")
models.shape

torch.Size([20, 3, 1207])

In [3]:
def center_models(models):
    """
    Remove center of mass from each model.
    
    Args:
        models: torch.Tensor of shape [num_models, 3, N]
                where 3 = (x, y, z) and N = number of atoms
    
    Returns:
        centered_models: torch.Tensor of same shape, centered at origin
    """
    # Compute center of mass for each model
    # Mean over atoms (dim=2) -> [num_models, 3]
    com = models.mean(dim=2, keepdim=True)  # [num_models, 3, 1]
    
    # Subtract center of mass
    centered_models = models - com
    
    return centered_models

# center models (just to be sure)
models = center_models(models)
# and save them in the current directory
torch.save(models, "models.pt")

In [ ]:
def visualize_conformations(models, indices=[0, 10, 19]):
    """Visualize selected conformations in 3D."""
    # Convert torch to numpy for plotting if needed
    if isinstance(models, torch.Tensor):
        models = models.numpy()
    
    fig = plt.figure(figsize=(20, 4))
    
    for plot_idx, conf_idx in enumerate(indices):
        ax = fig.add_subplot(1, 5, plot_idx + 1, projection='3d')
        
        coords = models[conf_idx].T  # [n_atoms, 3]
        ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], 
                  c=np.arange(len(coords)), cmap='viridis', s=20, alpha=0.6)
        
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        
        # Set equal aspect ratio
        max_range = np.array([coords[:, 0].max()-coords[:, 0].min(),
                             coords[:, 1].max()-coords[:, 1].min(),
                             coords[:, 2].max()-coords[:, 2].min()]).max() / 2.0
        mid_x = (coords[:, 0].max()+coords[:, 0].min()) * 0.5
        mid_y = (coords[:, 1].max()+coords[:, 1].min()) * 0.5
        mid_z = (coords[:, 2].max()+coords[:, 2].min()) * 0.5
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    plt.tight_layout()
    plt.show()
    
# Visualize (automatically converts to numpy for plotting)
visualize_conformations(models)

### Look at the parameters for image generation

During NLE flow training, we will generate images using the class `CryoEmSimulator`. 
This class uses as input a config file with the simulation parameters, including the models to use (the centered models stored in `models.pt` and the range spanned by all the cryo-EM parameters, such as sigma, defocus, SNR, ... The config file used here is called `simulation_parameters.json`.

In this test, we generate images with variable SNR, defocus, and other parameters in order to challenge our approach with realistic raw particles.

```
simulation_parameters.json

{
    "N_PIXELS": 128,
    "PIXEL_SIZE": 1.5,
    "SIGMA": [0.5, 5.0],
    "MODEL_FILE": "models.pt",
    "SHIFT": 0.0,
    "DEFOCUS": [0.5, 2.0],
    "SNR": [0.01, 0.5],
    "AMP": 0.1,
    "B_FACTOR": [1.0, 100.0]
}
```

### Pre-training of RESNET18 image embedding

Before proceeding we need to pre-train the image embedding. In order to do so, we need to use the script currently called `pretrain_resnet18-v4.py`. A typical command line is:

```
python pretrain_resnet18-v4.py --embedding RESNET18 \
                               --epochs 100 --batch_size 256 --device cuda:0 \
                               --output pretrained_resnet18.pt --embedding_dim 256 \ 
                               --image_config simulation_parameters.json --lr 0.0002 \
                               --simulation_batch_size 1024
```

where we can choose either `RESNET18` or `RESNET18_FFT_FILTER` as embedding and modify the number of images generated per batch with `--simulation_batch_size` (default is 1024).

The goal here is to create an embedding for the images that, in 256 dimensions, captures the information contained in the raw images and allows to distiguish different images in the embedded space. This embedding will be frozen (or minimally fine-tuned) during subsequent training of the NLE flow to avoid collapsing. Below is an example of output of the pre-training. Don't worry about the warning "reconstruction error", we want first of all the embedding to "diversify" different images.

### Train cryoSBI likelihood

We will now train the cryoSBI likelihood. The training is done with the function `nle_train_no_saving` which simulates images and simultaneously trains the likelihood. The function takes as input the config file `training_parameters.json` which contains the training and neural network parameters. The function also takes as input the config file `simulation_parameters.json` which contains the simulation parameters used to simulate the images. Finally, this class reads the pre-trained image embedding stored in `pretrained_resnet18.pt`.


```
training_parameters_nle.json

{
    "EMBEDDING": "RESNET18",
    "OUT_DIM": 256,
    "NUM_TRANSFORM": 5,
    "NUM_HIDDEN_FLOW": 2,
    "HIDDEN_DIM_FLOW": 256,
    "MODEL": "NSF",
    "LEARNING_RATE": 1e-3,
    "CLIP_GRADIENT": 1.0,
    "THETA_SHIFT": 9.5,
    "THETA_SCALE": 9.5,
    "BATCH_SIZE": 512
}
```


In [ ]:
train_nle_model.nle_train_no_saving(
    "simulation_parameters.json",
    "training_parameters_nle.json",
    200,
    "tutorial_estimator.pt",  # name of the estimator file
    "tutorial.loss",  # name of the loss file
    n_workers=4,  # number of workers for data loading
    device="cuda:0",  # device to use for training and simulation
    pretrained_embedding_path="pretrained_resnet18.pt",  # Your saved weights
    freeze_embedding=True,  # FREEZE IT - don't let it collapse!
    #freeze_embedding=False,  # Allow fine-tuning
    #use_differential_lr=True,  # Lower LR for embedding
    #embedding_lr_factor=0.01,  # 100x lower than flow LR
    saving_frequency=100,  # frequency of saving the model
    simulation_batch_size=2048,  # batch size for simulation
)

In [ ]:
plt.plot(torch.load("tutorial.loss"))
plt.xlabel("Epoch")
plt.ylabel("Loss")